# Centrality vs Efficient – Comparative Analysis

This notebook compares evacuation performance across four strategies:
- low awareness + efficient
- low awareness + centrality
- high awareness + efficient
- high awareness + centrality

Goal:
Assess whether centrality-based routing provides systematic advantages over efficient routing,
and whether such advantages depend on the awareness level.

Input:
One or more CSV files exported from `experiment_metrics` from `results\CSV`.

In [52]:
import os
import glob
import numpy as np
import pandas as pd

# Optional (pretty display in notebooks)
from IPython.display import display

## 1) Load one CSV (edit path)


In [53]:
path = "../results/CSV/Cruise_Ship_experiment_metrics.csv"
# path = "../results/CSV/Theme Park_experiment_metrics.csv"
# path = "../results/CSV/corridor_experiment_metrics.csv"

df = pd.read_csv(path)
display(df.head())


,experiment_id,case_name,agent_group_id,algorithm,awareness,n_records,mean_remaining_path_risk,remaining_path_risk_var,cumulative_risk_exposure,avg_path_length,min_time,avg_time,median_time,p90_time,max_time,case_name.1,risk_nodes,source_nodes,agents_per_source,random_seed
0,1,corridor_case_1,16,Centrality,High,17,0.0,0.0,0.0,3.470588,19.59,22.278,22.29,24.42,24.96,corridor_case_1,[],"[""16""]",[5],1001
1,1,corridor_case_1,16,Centrality,Low,17,0.0,0.0,0.0,3.470588,19.59,22.278,22.29,24.42,24.96,corridor_case_1,[],"[""16""]",[5],1001
2,1,corridor_case_1,16,Efficient,High,33,0.0,0.0,0.0,2.515152,19.59,22.278,22.29,24.42,24.96,corridor_case_1,[],"[""16""]",[5],1001
3,1,corridor_case_1,16,Efficient,Low,33,0.0,0.0,0.0,2.515152,19.59,22.278,22.29,24.42,24.96,corridor_case_1,[],"[""16""]",[5],1001
4,2,corridor_case_2,16,Centrality,High,26,0.0,0.0,0.0,4.961538,32.46,35.448,35.46,38.01,38.28,corridor_case_2,[],"[""16""]",[10],1002


## 2) Inspect columns (debug cell)

In [54]:
print("Columns in the dataset:")
for col in df.columns:
    print(f"- {col}")

Columns in the dataset:
- experiment_id
- case_name
- agent_group_id
- algorithm
- awareness
- n_records
- mean_remaining_path_risk
- remaining_path_risk_var
- cumulative_risk_exposure
- avg_path_length
- min_time
- avg_time
- median_time
- p90_time
- max_time
- case_name.1
- risk_nodes
- source_nodes
- agents_per_source
- random_seed


## 3) Basic cleanup & normalization
- Remove duplicated join artifact: `case_name.1` if present
- Normalize text fields (lowercase) for stable pivoting
- Ensure numeric KPI columns are numeric

In [55]:
# Drop duplicated case_name column created by joins
if "case_name.1" in df.columns:
    df = df.drop(columns=["case_name.1"])

# Normalize algorithm/awareness to lowercase (safe even if already lowercase)
df["algorithm"] = df["algorithm"].astype(str).str.strip().str.lower()
df["awareness"] = df["awareness"].astype(str).str.strip().str.lower()
df["case_name"] = df["case_name"].astype(str).str.strip()

# Build strategy label (you already have "strategy" column, but we rebuild it deterministically)
df["strategy"] = df["awareness"] + " awareness + " + df["algorithm"]

# Ensure numeric columns are numeric (coerce errors to NaN)
numeric_cols = [
    "n_records",
    "mean_remaining_path_risk",
    "remaining_path_risk_var",
    "cumulative_risk_exposure",
    "avg_path_length",
    "min_time",
    "avg_time",
    "median_time",
    "p90_time",
    "max_time",
    "random_seed",
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

display(df.head())

,experiment_id,case_name,agent_group_id,algorithm,awareness,n_records,mean_remaining_path_risk,remaining_path_risk_var,cumulative_risk_exposure,avg_path_length,min_time,avg_time,median_time,p90_time,max_time,risk_nodes,source_nodes,agents_per_source,random_seed,strategy
0,1,corridor_case_1,16,centrality,high,17,0.0,0.0,0.0,3.470588,19.59,22.278,22.29,24.42,24.96,[],"[""16""]",[5],1001,high awareness + centrality
1,1,corridor_case_1,16,centrality,low,17,0.0,0.0,0.0,3.470588,19.59,22.278,22.29,24.42,24.96,[],"[""16""]",[5],1001,low awareness + centrality
2,1,corridor_case_1,16,efficient,high,33,0.0,0.0,0.0,2.515152,19.59,22.278,22.29,24.42,24.96,[],"[""16""]",[5],1001,high awareness + efficient
3,1,corridor_case_1,16,efficient,low,33,0.0,0.0,0.0,2.515152,19.59,22.278,22.29,24.42,24.96,[],"[""16""]",[5],1001,low awareness + efficient
4,2,corridor_case_2,16,centrality,high,26,0.0,0.0,0.0,4.961538,32.46,35.448,35.46,38.01,38.28,[],"[""16""]",[10],1002,high awareness + centrality


## 4) Sanity checks
Ensure the four expected combinations exist:
- low/efficient, low/centrality, high/efficient, high/centrality

In [56]:
expected = {
    ("low", "efficient"),
    ("low", "centrality"),
    ("high", "efficient"),
    ("high", "centrality"),
}
present = set(zip(df["awareness"], df["algorithm"]))
missing = expected - present

print("Present (awareness, algorithm) pairs:", sorted(present))
if missing:
    print("⚠️ Missing expected pairs:", sorted(missing))
else:
    print("✅ All four expected strategy combinations are present.")

Present (awareness, algorithm) pairs: [('high', 'centrality'), ('high', 'efficient'), ('low', 'centrality'), ('low', 'efficient')]
✅ All four expected strategy combinations are present.


## 5) Global summary (aggregate across all cases)
This answers: "Overall, which strategy tends to be better?"

In [57]:
summary_global = (
    df
    .groupby("strategy", as_index=False)
    .agg(
        n_rows=("case_name", "size"),
        n_cases=("case_name", "nunique"),
        avg_time_mean=("avg_time", "mean"),
        avg_time_median=("avg_time", "median"),
        median_time_mean=("median_time", "mean"),
        p90_time_mean=("p90_time", "mean"),
        max_time_mean=("max_time", "mean"),
        min_time_mean=("min_time", "mean"),
        risk_exposure_mean=("cumulative_risk_exposure", "mean"),
        remaining_risk_mean=("mean_remaining_path_risk", "mean"),
    )
    .sort_values(["avg_time_mean", "p90_time_mean"], ascending=True)
)

display(summary_global)

,strategy,n_rows,n_cases,avg_time_mean,avg_time_median,median_time_mean,p90_time_mean,max_time_mean,min_time_mean,risk_exposure_mean,remaining_risk_mean
0,high awareness + centrality,13,10,31.295427,34.2564,31.324615,35.054769,36.295385,25.950000,0.035833,0.002464
1,high awareness + efficient,13,10,31.295427,34.2564,31.324615,35.054769,36.295385,25.950000,0.035833,0.002385
2,low awareness + centrality,13,10,31.858485,34.2564,31.896923,35.494615,36.653077,26.559231,0.040760,0.008798
3,low awareness + efficient,13,10,31.858485,34.2564,31.896923,35.494615,36.653077,26.559231,0.040760,0.008729


## 6) Case-wise comparison (centrality vs efficient within each awareness)
This answers: "In case X, was centrality better than efficient? Does the trend repeat?"

In [58]:
# Pivot per case. Values are per-row; if you have multiple groups per case, you can
# switch to median/mean aggregation here by setting aggfunc.
pivot_avg = df.pivot_table(
    index="case_name",
    columns=["awareness", "algorithm"],
    values="avg_time",
    aggfunc="mean"
)

display(pivot_avg.head())

awareness              high                  low          
algorithm        centrality efficient centrality efficient
case_name                                                 
corridor_case_1    22.27800  22.27800    22.2780   22.2780
corridor_case_10   37.35975  37.35975    37.8975   37.8975
corridor_case_2    35.44800  35.44800    35.4480   35.4480
corridor_case_3    38.71050  38.71050    38.7105   38.7105
corridor_case_4    41.37500  41.37500    43.2610   43.2610

In [59]:
# Add boolean indicators: centrality better than efficient
# (lower time is better)
pivot_avg["low: centrality better"] = pivot_avg[("low", "centrality")] < pivot_avg[("low", "efficient")]
pivot_avg["high: centrality better"] = pivot_avg[("high", "centrality")] < pivot_avg[("high", "efficient")]

# Also compute deltas (efficient - centrality): positive means centrality is better (smaller time)
pivot_avg["low: Δ(efficient - centrality)"] = pivot_avg[("low", "efficient")] - pivot_avg[("low", "centrality")]
pivot_avg["high: Δ(efficient - centrality)"] = pivot_avg[("high", "efficient")] - pivot_avg[("high", "centrality")]

display(pivot_avg)

awareness              high                  low            \
algorithm        centrality efficient centrality efficient   
case_name                                                    
corridor_case_1    22.27800  22.27800    22.2780   22.2780   
corridor_case_10   37.35975  37.35975    37.8975   37.8975   
corridor_case_2    35.44800  35.44800    35.4480   35.4480   
corridor_case_3    38.71050  38.71050    38.7105   38.7105   
corridor_case_4    41.37500  41.37500    43.2610   43.2610   
corridor_case_5    20.46000  20.46000    20.4600   20.4600   
corridor_case_6    34.25640  34.25640    34.2564   34.2564   
corridor_case_7    20.80300  20.80300    20.8030   20.8030   
corridor_case_8    41.51775  41.51775    45.8760   45.8760   
corridor_case_9    35.66640  35.66640    35.6664   35.6664   

awareness        low: centrality better high: centrality better  \
algorithm                                                         
case_name                                                         
corridor_case_1                   False                   False   
corridor_case_10                  False                   False   
corridor_case_2                   False                   False   
corridor_case_3                   False                   False   
corridor_case_4                   False                   False   
corridor_case_5                   False                   False   
corridor_case_6                   False                   False   
corridor_case_7                   False                   False   
corridor_case_8                   False                   False   
corridor_case_9                   False                   False   

awareness        low: Δ(efficient - centrality)  \
algorithm                                         
case_name                                         
corridor_case_1                             0.0   
corridor_case_10                            0.0   
corridor_case_2                             0.0   
corridor_case_3                             0.0   
corridor_case_4                             0.0   
corridor_case_5                             0.0   
corridor_case_6                             0.0   
corridor_case_7                             0.0   
corridor_case_8                             0.0   
corridor_case_9                             0.0   

awareness        high: Δ(efficient - centrality)  
algorithm                                         
case_name                                         
corridor_case_1                              0.0  
corridor_case_10                             0.0  
corridor_case_2                              0.0  
corridor_case_3                              0.0  
corridor_case_4                              0.0  
corridor_case_5                              0.0  
corridor_case_6                              0.0  
corridor_case_7                              0.0  
corridor_case_8                              0.0  
corridor_case_9                              0.0

## 7) Trend counts (how often centrality wins)
This is a strong "study result" summary.

In [60]:
trend_summary = pd.DataFrame(
    {
        "centrality better (low awareness)": [
            int(pivot_avg["low: centrality better"].sum()),
            int((~pivot_avg["low: centrality better"]).sum()),
        ],
        "centrality better (high awareness)": [
            int(pivot_avg["high: centrality better"].sum()),
            int((~pivot_avg["high: centrality better"]).sum()),
        ],
    },
    index=["yes", "no"],
)

display(trend_summary)

,centrality better (low awareness),centrality better (high awareness)
yes,0,0
no,10,10


## 8) Case-wise ranking table (which strategy is best per case)
This gives you "best strategy per case" and lets you see if one dominates.


In [61]:
# Create a strategy-wise pivot for avg_time
pivot_strategy = df.pivot_table(
    index="case_name",
    columns="strategy",
    values="avg_time",
    aggfunc="mean"
)

# Rank strategies per case (1 = best/lowest time)
ranked = pivot_strategy.rank(axis=1, method="min", ascending=True)

# Identify best strategy label per case
best_strategy = pivot_strategy.idxmin(axis=1)
best_time = pivot_strategy.min(axis=1)

ranking_table = pivot_strategy.copy()
ranking_table["best_strategy"] = best_strategy
ranking_table["best_avg_time"] = best_time

display(ranking_table.sort_values("best_avg_time").head(20))

strategy,high awareness + centrality,high awareness + efficient,low awareness + centrality,low awareness + efficient,best_strategy,best_avg_time
case_name,,,,,,
corridor_case_5,20.46000,20.46000,20.4600,20.4600,high awareness + centrality,20.46000
corridor_case_7,20.80300,20.80300,20.8030,20.8030,high awareness + centrality,20.80300
corridor_case_1,22.27800,22.27800,22.2780,22.2780,high awareness + centrality,22.27800
corridor_case_6,34.25640,34.25640,34.2564,34.2564,high awareness + centrality,34.25640
corridor_case_2,35.44800,35.44800,35.4480,35.4480,high awareness + centrality,35.44800
corridor_case_9,35.66640,35.66640,35.6664,35.6664,high awareness + centrality,35.66640
corridor_case_10,37.35975,37.35975,37.8975,37.8975,high awareness + centrality,37.35975
corridor_case_3,38.71050,38.71050,38.7105,38.7105,high awareness + centrality,38.71050
corridor_case_4,41.37500,41.37500,43.2610,43.2610,high awareness + centrality,41.37500


## 9) Summary of best-strategy frequency
"How many cases does each strategy win?"


In [62]:
best_strategy_counts = best_strategy.value_counts().rename_axis("strategy").reset_index(name="wins")
display(best_strategy_counts)

,strategy,wins
0,high awareness + centrality,10


## 10) Compare by another KPI (risk exposure)
Same structure, but using `cumulative_risk_exposure` (lower is better).


In [63]:
pivot_risk = df.pivot_table(
    index="case_name",
    columns=["awareness", "algorithm"],
    values="cumulative_risk_exposure",
    aggfunc="mean"
)

# Centrality better if it reduces risk exposure
pivot_risk["low: centrality better (risk)"] = pivot_risk[("low", "centrality")] < pivot_risk[("low", "efficient")]
pivot_risk["high: centrality better (risk)"] = pivot_risk[("high", "centrality")] < pivot_risk[("high", "efficient")]

# Delta risk
pivot_risk["low: Δrisk(efficient - centrality)"] = pivot_risk[("low", "efficient")] - pivot_risk[("low", "centrality")]
pivot_risk["high: Δrisk(efficient - centrality)"] = pivot_risk[("high", "efficient")] - pivot_risk[("high", "centrality")]

display(pivot_risk)


awareness              high                  low            \
algorithm        centrality efficient centrality efficient   
case_name                                                    
corridor_case_1    0.000000  0.000000   0.000000  0.000000   
corridor_case_10   0.066532  0.066532   0.069297  0.069297   
corridor_case_2    0.000000  0.000000   0.000000  0.000000   
corridor_case_3    0.194032  0.194032   0.194032  0.194032   
corridor_case_4    0.009495  0.009495   0.035253  0.035253   
corridor_case_5    0.000000  0.000000   0.000000  0.000000   
corridor_case_6    0.065333  0.065333   0.065333  0.065333   
corridor_case_7    0.000000  0.000000   0.000000  0.000000   
corridor_case_8    0.015071  0.015071   0.047838  0.047838   
corridor_case_9    0.048828  0.048828   0.048828  0.048828   

awareness        low: centrality better (risk) high: centrality better (risk)  \
algorithm                                                                       
case_name                                                                       
corridor_case_1                          False                          False   
corridor_case_10                         False                          False   
corridor_case_2                          False                          False   
corridor_case_3                          False                          False   
corridor_case_4                          False                          False   
corridor_case_5                          False                          False   
corridor_case_6                          False                          False   
corridor_case_7                          False                          False   
corridor_case_8                          False                          False   
corridor_case_9                          False                          False   

awareness        low: Δrisk(efficient - centrality)  \
algorithm                                             
case_name                                             
corridor_case_1                                 0.0   
corridor_case_10                                0.0   
corridor_case_2                                 0.0   
corridor_case_3                                 0.0   
corridor_case_4                                 0.0   
corridor_case_5                                 0.0   
corridor_case_6                                 0.0   
corridor_case_7                                 0.0   
corridor_case_8                                 0.0   
corridor_case_9                                 0.0   

awareness        high: Δrisk(efficient - centrality)  
algorithm                                             
case_name                                             
corridor_case_1                                  0.0  
corridor_case_10                                 0.0  
corridor_case_2                                  0.0  
corridor_case_3                                  0.0  
corridor_case_4                                  0.0  
corridor_case_5                                  0.0  
corridor_case_6                                  0.0  
corridor_case_7                                  0.0  
corridor_case_8                                  0.0  
corridor_case_9                                  0.0

In [64]:
trend_risk_summary = pd.DataFrame(
    {
        "centrality reduces risk (low awareness)": [
            int(pivot_risk["low: centrality better (risk)"].sum()),
            int((~pivot_risk["low: centrality better (risk)"]).sum()),
        ],
        "centrality reduces risk (high awareness)": [
            int(pivot_risk["high: centrality better (risk)"].sum()),
            int((~pivot_risk["high: centrality better (risk)"]).sum()),
        ],
    },
    index=["yes", "no"],
)

display(trend_risk_summary)

,centrality reduces risk (low awareness),centrality reduces risk (high awareness)
yes,0,0
no,10,10
